# Prepare Training Data

**Next:** [02-submit-training-job.ipynb](02-submit-training-job.ipynb)

---

This notebook validates, preprocesses, and uploads training data for fine-tuning small language models on Azure ML.

In [ ]:
# Import required modules
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.data.preprocessing import (
    validate_jsonl_format,
    load_jsonl_data,
    split_train_validation,
    calculate_data_statistics,
    save_jsonl_data,
)
from src.data.upload_to_blob import (
    upload_file_to_blob,
    upload_directory_to_blob,
)
from src.utils.config import load_config
from src.utils.logging_config import setup_logging

# Setup logging
logger = setup_logging(log_level="INFO")

## 1. Load Configuration

In [ ]:
# Load configuration
config = load_config()

print(f"Azure Subscription: {config.azure.subscription_id}")
print(f"Resource Group: {config.azure.resource_group}")
print(f"Storage Account: {config.storage.account_name}")
print(f"Container: {config.storage.container_name}")

## 2. Validate Training Data

In [ ]:
# Path to your training data (default location)
input_file = Path("../data/training_data.jsonl")

# Check if file exists
if not input_file.exists():
    print(f"❌ Training data not found at: {input_file}")
    print("\n💡 Quick fix options:")
    print("1. Place your training data at: data/training_data.jsonl")
    print("2. Use the example: cp ../data/training_data.jsonl.example ../data/training_data.jsonl")
    print("3. Update the path above to point to your JSONL file")
    raise FileNotFoundError(f"Training data not found: {input_file}")

print(f"📂 Using training data: {input_file}")

# Validate the format
is_valid, errors = validate_jsonl_format(input_file)

if is_valid:
    print("✅ Data validation successful!")
else:
    print("❌ Data validation failed:")
    for error in errors[:10]:  # Show first 10 errors
        print(f"  - {error}")
    if len(errors) > 10:
        print(f"  ... and {len(errors) - 10} more errors")
    raise ValueError("Data validation failed. Please fix the errors above.")

## 3. Load and Inspect Data

In [ ]:
# Load the data
data = load_jsonl_data(input_file)

print(f"Loaded {len(data)} samples")
print("\nFirst sample:")
print(f"Prompt: {data[0]['prompt'][:100]}...")
print(f"Completion: {data[0]['completion'][:100]}...")

## 4. Calculate Statistics

In [ ]:
# Calculate dataset statistics
stats = calculate_data_statistics(data)

print("Dataset Statistics:")
print(f"  Number of samples: {stats['num_samples']}")
print(f"  Avg prompt length: {stats['avg_prompt_length']:.1f} chars")
print(f"  Avg completion length: {stats['avg_completion_length']:.1f} chars")
print(f"  Total tokens (estimate): {stats['total_tokens_estimate']:,}")

## 5. Split into Train/Validation Sets

In [ ]:
# Split the data
train_data, val_data = split_train_validation(
    data,
    validation_split=0.2,
    seed=42
)

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

# Calculate stats for each split
train_stats = calculate_data_statistics(train_data)
val_stats = calculate_data_statistics(val_data)

## 6. Save Split Data Locally

In [ ]:
# Create output directory
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save train and validation files
train_file = output_dir / "train.jsonl"
val_file = output_dir / "val.jsonl"

save_jsonl_data(train_data, train_file)
save_jsonl_data(val_data, val_file)

print(f"✅ Saved training data to {train_file}")
print(f"✅ Saved validation data to {val_file}")

## 7. Upload to Azure Blob Storage

In [ ]:
# Upload train and validation files to blob storage
train_url = upload_file_to_blob(
    train_file,
    "training-data/train.jsonl",
    config.storage,
    overwrite=True
)

val_url = upload_file_to_blob(
    val_file,
    "training-data/val.jsonl",
    config.storage,
    overwrite=True
)

print(f"✅ Training data uploaded to: {train_url}")
print(f"✅ Validation data uploaded to: {val_url}")

## Summary

✅ **Training data successfully prepared and uploaded!**

Your data is now ready for remote training on Azure ML. The validated and split datasets are available in your Azure ML datastore.

---

## Navigation

**Next:** [02-submit-training-job.ipynb](02-submit-training-job.ipynb)